# Stakes diagnostics

Sanity-check the chamber-control Monte Carlo before trusting the stakes scores in the combined Impact Score:

1. **Chamber composition distribution** — how many D seats per iteration?
2. **Stakes histogram** — most seats should have stakes ≈ 0; pivotal seats fill the tails.
3. **Top 10 highest-stakes 2024 races** — should be recognizable battlegrounds (PA, NY, MI, AZ, NV).
4. **Quantile crossing** — diagnostic from the multi-quantile model that feeds the MC.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, '../src')
from oath_score.scores.multi_quantile import MultiQuantileCompetitiveness
from oath_score.scores.chamber import build_chamber
from oath_score.scores.stakes import (
    PIVOTAL_THRESHOLD, StakesSimulator, sigma_for_snapshot,
)

PROC = Path('../data/processed')
RAW = Path('../data/raw')
train = pd.concat([pd.read_parquet(PROC / f'candidates_{c}_T-60.parquet') for c in (2014, 2016, 2022)],
                  ignore_index=True)
test = pd.read_parquet(PROC / 'candidates_2024_T-60.parquet')

model = MultiQuantileCompetitiveness(feature_set_name='full').fit(train)
scored = model.score(test)
dems = scored.loc[scored['party_major'] == 'D'].copy()
quants = model.predict_quantiles(dems)

chamber = build_chamber(2024, RAW)
uncontested_d = chamber.deterministic_d_count()
print(f'2024 chamber: {chamber.n_seats} seats, {chamber.n_d_locks} d_locks, {chamber.n_r_locks} r_locks')
print(f'Uncontested D baseline: {uncontested_d}')
print(f'Scoring {len(dems)} contested-race D candidates')

sim = StakesSimulator(sigma=sigma_for_snapshot('T-60'), n_iter=10000, chamber_threshold=None)
result = sim.simulate(quants, np.array(model.quantiles), uncontested_d_count=uncontested_d)

## 1. Chamber composition distribution

Distribution of total D seats across the 10k MC iterations. The dynamic threshold is the median of this distribution; iterations above that count as "D control" for the stakes computation.

In [ ]:
total_d = result.seat_d_won.sum(axis=1) + uncontested_d
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(total_d, bins=30, color='#3366cc', alpha=0.8)
median = np.median(total_d)
ax.axvline(median, color='red', linestyle='--', label=f'median (dynamic threshold): {median:.0f}')
ax.axvline(218, color='black', linestyle=':', alpha=0.5, label='literal majority: 218')
ax.set_xlabel('Total D-seat count')
ax.set_ylabel('MC iterations')
ax.set_title(f'2024 T-60 chamber-D distribution (10k iter, σ=3pp)\nP(D >= median) = {result.chamber_d_rate:.3f}')
ax.legend()
plt.tight_layout()
print(f'Mean total D: {total_d.mean():.1f}, std: {total_d.std():.1f}')
print(f'Range: [{total_d.min()}, {total_d.max()}]')
print(f'P(D >= 218 literal): {(total_d >= 218).mean():.3f}')

## 2. Stakes histogram

Most seats should have stakes ≈ 0 — the chamber outcome doesn't depend on them. A few hundred lopsided races contribute essentially nothing to chamber uncertainty. The pivotal tail is what matters for donor allocation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(result.stakes_raw, bins=40, color='#3366cc', alpha=0.8)
axes[0].axvline(PIVOTAL_THRESHOLD, color='red', linestyle='--', alpha=0.6, label=f'pivotal threshold ±{PIVOTAL_THRESHOLD}')
axes[0].axvline(-PIVOTAL_THRESHOLD, color='red', linestyle='--', alpha=0.6)
axes[0].set_xlabel('Stakes (raw, signed)')
axes[0].set_ylabel('Count of seats')
axes[0].set_title('Per-seat stakes distribution (2024 T-60)')
axes[0].legend()
axes[1].hist(result.stakes_normalized, bins=20, color='#cc6633', alpha=0.8)
axes[1].set_xlabel('Stakes (min-max normalized)')
axes[1].set_ylabel('Count of seats')
axes[1].set_title('Normalized stakes (input to combined score)')
plt.tight_layout()
print(f'Pivotal seats (|raw|>{PIVOTAL_THRESHOLD}): {result.n_pivotal} of {len(result.stakes_raw)}')

## 3. Top 10 highest-stakes 2024 races

Sanity check: are the highest-stakes races ones a political reporter would recognize as battleground 2024 districts? If we see PA, MI, NY, AZ, NV, NC tossups, the simulator is producing sensible output.

In [ ]:
dems = dems.copy()
dems['stakes_raw'] = result.stakes_raw
dems['stakes_normalized'] = result.stakes_normalized
top = dems.nlargest(10, 'stakes_normalized')[
    ['state_abbr', 'district', 'last_name', 'cook_rating', 'margin_pct', 'stakes_raw', 'stakes_normalized']
]
top.style.format({
    'cook_rating': '{:.1f}', 'margin_pct': '{:+.3f}',
    'stakes_raw': '{:+.4f}', 'stakes_normalized': '{:.3f}',
}).set_caption('Top 10 highest-stakes 2024 Dem candidates (full feature set, T-60)')

## 4. Multi-quantile crossing rate

The multi-quantile model's nine independent regressions can produce out-of-order quantile predictions. The MC repairs them with a per-row sort before sampling, but the crossing rate is a useful indicator of whether the underlying fits are stable.

In [ ]:
_ = model.predict_proba(test)  # populate crossing_rate diagnostic
print(f'Quantile crossing rate (full / 2024): {model.crossing_rate:.3f}')
print(f'  Of {len(test)} test rows, ~{int(model.crossing_rate * len(test))} had at least one out-of-order pair.')
print('  Crossings are repaired in-place by per-row sort before sampling.')